# シミュレーション計算実行用ノートブック

## 1. 今回の実験の説明

In [ ]:
DESCRIPTION = '''
python bindingの機能を使ってlammpsをlumeから実行する実験
'''
ISSUE_NO = ''
EXEC_NAME = 'Ax'
RUN_SCRIPT = "calcLammps"

PREV_RUNID = ''
RESTART_FROM = ''

MLFLOW_EXP_TYPE = "LumeLammpsDev2"


## 2. シミュレーションパラメータ

In [ ]:
import importlib, json
sim = importlib.import_module(f'scripts.{RUN_SCRIPT}')

sim_params = sim.default_prams

## 必要なら適宜修正する
sim_params["data_file"] = 'data.Ax_lmp_in'
sim_params["input_file"] = 'in.Ax_lmp_in'
sim_params["log_file"] = f'{EXEC_NAME}.txt'
sim_params["data_dir"] = f'data/{EXEC_NAME}'
sim_params["run_steps"] = 10000


print(json.dumps(sim_params, indent=4, ensure_ascii=False))

dump_files = []
snapshots = "lammps_snap"
restart_files = []

## 3. mlflow変数

In [ ]:
import mlflow
import os
from time import strftime, gmtime
from utils.info_utils import MLflowEnvLogger

In [ ]:
ROOT = os.getenv("HOME")

## 1台構成の時
#MLFLOW_TRACKING_URI = f"sqlite:///{ROOT}/mlruns/mlflow.db"
#MLFLOW_STORAGE = f"file://{ROOT}/mlstorage"
### S3 bucket を指定する場合
MLFLOW_STORAGE = f"s3://{os.getenv('S3STORAGEBUCKET', 'my-mlflow-artifact-s3-bucket')}/mlstorage/"

## mlflow serverのIPを指定
MLFLOW_TRACKING_URI = "http://localhost:5000"


## github, backlogなどでチケット管理をしている場合はそのBASE URLを設定
ISSUE_BASE_URL = 'https://xxxxx/'

### dump fileの圧縮に使うコマンド（pixz があれば推奨）
ARCHIVE_COMMAND = "pixz"

###
### mlflow変数　自動設定
###
MYNAME = os.getenv("USER")
#GIT_INFO = gitutils.get_info()
RUN_NAME = EXEC_NAME + strftime("-%Y-%m-%d-%H-%M-%S", gmtime())

if ISSUE_NO != '':
    ISSUE_NAME = f'\n[{ISSUE_NO}]({ISSUE_BASE_URL}{ISSUE_NO})'
else:
    ISSUE_NAME = ''


## 4. シミュレーション実行

In [ ]:
###
### mlflow処理開始
###
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_exp = mlflow.get_experiment_by_name(MLFLOW_EXP_TYPE)
if mlflow_exp is None:
    mlflow_exp_id = mlflow.create_experiment(name=MLFLOW_EXP_TYPE, artifact_location=MLFLOW_STORAGE)
else:
    mlflow_exp_id = mlflow_exp.experiment_id

In [ ]:
mlflow_run = mlflow.start_run(
    experiment_id=mlflow_exp_id,
    run_name=RUN_NAME,
    description=f'{DESCRIPTION}{ISSUE_NAME}',
    log_system_metrics=True)

print(f"Run ID: {mlflow_run.info.run_id}")

mlflow.set_tag("mlflow.user", MYNAME)
mlflow.set_tag("simulation", EXEC_NAME)
mlflow.set_tag("run_script", RUN_SCRIPT)
#mlflow.log_params({'git_commit': GIT_INFO['commit'], 'git_branch': GIT_INFO['branch']})
#mlflow.log_artifact('git.diff.txt', artifact_path='git_info')

In [ ]:
##
## シミュレーション処理　シミュレーション初期化
##
import sys

sim.log_GPU_info(mlflow.set_tags)
mlflow.log_params(sim_params)
mlflow.log_params({
    "dump_files": dump_files,
    "snapshots": snapshots,
    "restart_files": restart_files,
})
mlflow.set_tags(MLflowEnvLogger.log_all_env_tags())

if PREV_RUNID:
    prev_state_path = mlflow.artifacts.download_artifacts(
        run_id=PREV_RUNID,
        artifact_path="restarts",    # 取得したいアーティファクト内のパス
    )
    files = os.listdir(prev_state_path)
    if RESTART_FROM != '':
        RESTART_FILE = RESTART_FROM
    elif len(files) > 0:
        RESTART_FILE = files[0]
    else:
        sys.exit(0)
    sim.load_previous_state(sim_params, f'{prev_state_path}/{RESTART_FILE}')
    os.system(f"rm -rf {prev_state_path}")
    mlflow.set_tag("PREV_RUNID", PREV_RUNID)
else:
    sim.create_initial_state(sim_params, mlflow.log_params, snapshots)

In [ ]:
##
## シミュレーション処理　メインループ
##
restart = f'restart.{RUN_NAME}'
sim.run(sim_params, mlflow.log_metrics, restart, dump_files)

In [ ]:
##
## シミュレーション処理　結果の登録・保存
##
artifacts = {
    "dumpfiles": dump_files,
    "snapshots": [snapshots],
    "restarts": [restart],
}
if "log_file" in sim_params:
    artifacts.update({"log": f'log.{sim_params["log_file"]}'})

sim.store_artifacts(artifacts, mlflow.log_artifact, f"{ARCHIVE_COMMAND} -t", cleanup=True)

mlflow.end_run()

In [ ]:
## 異常・中断時の mlflow.end_run()
run_info = mlflow.get_run(mlflow_run.info.run_id)
if run_info.info.lifecycle_stage == "active":
    mlflow.end_run(status='KILLED')
